In [42]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from helpers.logger import log_step, save_log
from helpers.data_loader import DataLoader
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

INTERIM_PATH = DataLoader.interim("merged_inspections_licenses_inner.csv")
OUTPUT_PATH = DataLoader.interim("cleaning/cleaning_00_output.csv")
LOG_PATH = DataLoader.interim("cleaning/cleaning_00_log.csv")

In [43]:
df = pd.read_csv(INTERIM_PATH)
print('Loaded shape:', df.shape)
log_step('Initial Load', df.shape[0], df.shape[0], df.shape[1], df.shape[1])
df.columns

Loaded shape: (196825, 59)


Index(['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results',
       'Violations', 'Latitude', 'Longitude', 'Location', 'Historical Wards 2003-2015', 'Zip Codes', 'Community Areas', 'Census Tracts', 'Wards', 'BL_ID', 'BL_LICENSE_ID',
       'ACCOUNT NUMBER', 'SITE NUMBER', 'BL_LEGAL_NAME', 'BL_DBA_NAME', 'BL_ADDRESS', 'BL_CITY', 'BL_STATE', 'BL_ZIP_CODE', 'WARD', 'PRECINCT', 'WARD PRECINCT', 'POLICE DISTRICT',
       'COMMUNITY AREA', 'COMMUNITY AREA NAME', 'NEIGHBORHOOD', 'LICENSE CODE', 'LICENSE DESCRIPTION', 'BUSINESS ACTIVITY ID', 'BUSINESS ACTIVITY', 'LICENSE NUMBER',
       'APPLICATION TYPE', 'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE', 'PAYMENT DATE', 'CONDITIONAL APPROVAL', 'LICENSE TERM START DATE',
       'LICENSE TERM EXPIRATION DATE', 'LICENSE APPROVED FOR ISSUANCE', 'DATE ISSUED', 'LICENSE STATUS', 'LICENSE STATUS CHANGE DATE', 'SSA', 'BL_LATITUD

In [44]:
COLS_TO_DROP = [
    # exact duplicates of main dataset columns
    'BL_ADDRESS', 'BL_CITY', 'BL_STATE', 'BL_ZIP_CODE',
    'BL_LATITUDE', 'BL_LONGITUDE', 'BL_LOCATION',
    'BL_LEGAL_NAME', 'BL_DBA_NAME',

    # administrative IDs with no predictive value
    'BL_ID', 'BL_LICENSE_ID', 'ACCOUNT NUMBER', 'SITE NUMBER',

    # political/administrative boundaries
    'WARD', 'PRECINCT', 'WARD PRECINCT', 'POLICE DISTRICT', 'SSA',

    # licensing process snapshots (not establishment state)
    'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE',
    'PAYMENT DATE', 'CONDITIONAL APPROVAL',
    'LICENSE APPROVED FOR ISSUANCE', 'DATE ISSUED',
    'LICENSE STATUS CHANGE DATE',

    'BUSINESS ACTIVITY ID', 'BUSINESS ACTIVITY',

    # join key — already served its purpose
    'LICENSE NUMBER', 'LICENSE CODE',

    # redundant with COMMUNITY AREA NAME
    'NEIGHBORHOOD', 'COMMUNITY AREA',
]

In [45]:
rows_before, cols_before = df.shape
df = df.drop(columns=COLS_TO_DROP)
rows_after, cols_after = df.shape
print(f"Dropped {cols_before - cols_after} columns: {cols_before} → {cols_after}")
log_step('drop_columns', rows_before, rows_after, cols_before, cols_after, note='; '.join(COLS_TO_DROP))


Dropped 31 columns: 59 → 28


In [46]:
df.columns


Index(['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results',
       'Violations', 'Latitude', 'Longitude', 'Location', 'Historical Wards 2003-2015', 'Zip Codes', 'Community Areas', 'Census Tracts', 'Wards', 'COMMUNITY AREA NAME',
       'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE', 'LICENSE STATUS'],
      dtype='object')

In [47]:
'''
RENEW — renewal of existing license
ISSUE — brand new license
C_LOC / C_SBA / C_EXPA / C_CAPA — change of location / doing-business-as name / expansion / capacity

C_* types together are only ~500 rows 
will collapse them into a single "change" category when we get to feature engineering
Three buckets: new, renewal, change
'''

df['APPLICATION TYPE'].value_counts()


APPLICATION TYPE
RENEW     166547
ISSUE      20067
C_LOC        268
C_SBA         94
C_EXPA        92
C_CAPA        59
Name: count, dtype: int64

In [48]:
'''
AAI — Active, Issued
AAC — Active, Conditional
REV — Revoked
REA — Revoked, Appeal pending

binary feature license_active (AAI/AAC → 1, REV/REA → 0)
'''

df['APPLICATION TYPE'].value_counts()


APPLICATION TYPE
RENEW     166547
ISSUE      20067
C_LOC        268
C_SBA         94
C_EXPA        92
C_CAPA        59
Name: count, dtype: int64

In [51]:
cols = [
    'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION',
    'APPLICATION TYPE', 'LICENSE TERM START DATE',
    'LICENSE TERM EXPIRATION DATE', 'LICENSE STATUS'
]
df[cols].isnull().sum().rename('null_count').to_frame().assign(
    null_pct=lambda x: (x['null_count'] / len(df) * 100).round(2)
)

,null_count,null_pct
COMMUNITY AREA NAME,11567,5.88
LICENSE DESCRIPTION,9698,4.93
APPLICATION TYPE,9698,4.93
LICENSE TERM START DATE,9761,4.96
LICENSE TERM EXPIRATION DATE,9714,4.94
LICENSE STATUS,9698,4.93


In [52]:
df['AKA Name'].value_counts()

AKA Name
SUBWAY               3281
DUNKIN DONUTS        1395
MCDONALD'S            771
7-ELEVEN              752
BURGER KING           376
                     ... 
OAK TERRACE             1
MANGO'S                 1
INDIAN GOURMET          1
CAGNEY'S                1
HOLE IN ONE PIZZA       1
Name: count, Length: 26346, dtype: int64

In [53]:
# === SAVE OUTPUTS ===
# Save cleaned data for next stage (cleaning_01)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'✓ Saved cleaned data to: {OUTPUT_PATH}')

# Save transformation log
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
save_log(LOG_PATH)
print(f'✓ Saved cleaning log to: {LOG_PATH}')


✓ Saved cleaned data to: ..\..\data\interim\cleaning\cleaning_00_output.csv
✓ Saved cleaning log to: ..\..\data\interim\cleaning\cleaning_00_log.csv
